# Laboratorio 2 · Extraer, comprobar e iterar

**Versión de trabajo para revisión docente.** Esta base usa los ejemplos sintéticos de la clase y arranca en modo simulado. El dominio y la muestra definitivos se incorporarán tras la votación; esta ejecución todavía no es una entrega evaluada ni una medición de modelos.

Objetivo: comparar dos instrucciones con el mismo esquema, documentos y modelo, medir exactitud por campo frente a una referencia humana y explicar un cambio y una falla restante. La salida incluye valor, confianza declarada y respaldo por campo.

Trabajen en el equipo del proyecto y alternen ejecución y revisión. El producto compartido vive en `labs/lab02/`; la reflexión, el historial del asistente y el enlace a la versión del equipo van al portafolio individual. Se reservan cinco minutos para la verificación individual al cierre; su lugar y el plazo de entrega los indicará el docente.

## 1. Entorno y modo de trabajo

En Codespaces o local se instala este código con el entorno del repositorio. Esta versión de trabajo aún no tiene una etiqueta pública `lab02`: la ruta de Colab se habilitará al publicar la versión revisada. No uses `lab01` para este notebook, porque no incluye las utilidades nuevas.

`simulated` usa salidas preparadas sin llamadas externas. Para probar un modelo real, cambia `MODE` a `real`, configura la clave en los secretos del entorno y vuelve a ejecutar desde aquí. `generation` se resuelve por configuración, sin fijar el identificador del modelo en el código.

In [ ]:
import asyncio
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
from typing import Generic, Literal, TypeVar

import pandas as pd
from IPython.display import Image, display
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from pydantic_ai import Agent, BinaryContent
from pydantic_ai.usage import UsageLimits

from curso_ia import (
    create_model, document_input, lab_resource, load_config, load_json,
    save_experiment, status,
)
from curso_ia.measure import cost

MODE = os.environ.get("CURSO_MODE", "simulated")
os.environ["CURSO_MODE"] = MODE
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
config = load_config(ROOT)
MODEL_ROLE = "generation"
model_ref = config.models[MODEL_ROLE]
print("Modo:", config.mode, "· Modelo configurado:", model_ref)
display(pd.DataFrame(status(config)))

## 2. Documentos y referencia humana

La base tiene seis mensajes y una imagen que representa el segundo mensaje: siete documentos, con un par texto/imagen. Este par no equivale a dos observaciones independientes sobre usuarios distintos. Los dos ejemplos de instrucción están separados de esos documentos.

Lee los originales y verifica las referencias antes de consultar el modelo. En esta versión didáctica fueron redactadas y revisadas al preparar la clase; la muestra definitiva necesitará su propia revisión. Mantén `REFERENCE_REVIEWED = False` hasta realizar tu revisión y documentar cualquier desacuerdo.

Si se solicita reemplazo y reembolso simultáneamente sin priorizar, este esquema de una sola acción conserva `null` y el conflicto se aclara con la persona. No se completa un monto que no aparece. Los valores cero explícitos sí se conservan.

In [ ]:
# Con la muestra definitiva, indica la ruta al manifiesto y a sus documentos.
DATASET_PATH = None  # Ejemplo: Path("data/lab02/cases.json")
manifest = DATASET_PATH if DATASET_PATH else lab_resource(2, "cases.json")
dataset = load_json(manifest)
cases = dataset["cases"]
REFERENCE_REVIEWED = False

print("Conjunto:", dataset["name"], "· Documentos:", len(cases))
print(dataset["status"])
display(pd.DataFrame([
    {"case_id": case["id"], "document": case["document"],
     **case["reference"], "annotation_note": case["annotation_note"]}
    for case in cases
]))

In [ ]:
for case in cases:
    path = (DATASET_PATH.parent / case["document"] if DATASET_PATH
            else lab_resource(2, case["document"]))
    content = document_input(path)
    print(case["id"], "·", case["document"])
    if isinstance(content, BinaryContent) and content.media_type.startswith("image/"):
        display(Image(data=content.data))
    elif isinstance(content, str):
        print(content)
    else:
        print("Documento binario:", content.media_type, "· revisa el original.")

## 3. Definir el esquema de salida

Cada campo debe estar presente, pero su `value` puede ser nulo. `declared_confidence` es lo que declara el modelo entre 0 y 1: no es una probabilidad calibrada de acertar. `evidence` contiene un fragmento de respaldo o queda nulo si falta información; que el esquema lo admita no comprueba que el fragmento sea pertinente.

`FieldValue` repite la misma estructura para tipos distintos. `Extraction` define los tres campos de esta demostración; se adaptará al dominio común. La validación de tipos es estricta para evitar confundir, por ejemplo, una cadena de texto o un booleano con un monto entero.

In [ ]:
T = TypeVar("T")

class FieldValue(BaseModel, Generic[T]):
    model_config = ConfigDict(extra="forbid", strict=True)
    value: T | None
    declared_confidence: float = Field(ge=0, le=1)
    evidence: str | None

class Extraction(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    order_id: FieldValue[str]
    requested_action: FieldValue[Literal["replacement", "refund"]]
    amount: FieldValue[int]

FIELDS = list(Extraction.model_fields)
print("Campos:", FIELDS)

## 4. Comparar instrucciones sin ejemplos y con ejemplos

Mantendremos el esquema, las reglas, el modelo y los documentos. La única modificación prevista es añadir los dos ejemplos de instrucción. Sus cifras de confianza son ilustrativas y no expresan una medición de fiabilidad. La confianza que declare el modelo se contrastará después con los aciertos.

Las referencias de los documentos evaluados no forman parte de la entrada al modelo. Si usas repetidamente estos casos para ajustar las instrucciones, pasan a servir como desarrollo: después comprueba con casos nuevos reservados.

In [ ]:
BASE_INSTRUCTION = """
Extrae order_id, requested_action y amount del documento usando el esquema.
Para cada campo devuelve value, declared_confidence y evidence.
Usa solo lo que el documento respalda. No inventes datos ni soluciones.
Acción: replacement, refund o null. Monto: pesos enteros o null.
Conserva null si el dato falta o hay una ambigüedad sin resolver.
Si se piden reemplazo y reembolso simultáneos sin priorizar, usa null en la acción.
No reemplaces un monto desconocido por cero ni un cero explícito por null.
Cuando haya respaldo, evidence debe citar un fragmento breve del documento;
si no hay respaldo, usa null. La confianza es tu declaración, no una garantía.
""".strip()

examples = []
for example in dataset["prompt_examples"]:
    example_output = {
        field: {"value": value, "declared_confidence": 0.5,
                "evidence": example["text"] if value is not None else None}
        for field, value in example["reference"].items()
    }
    examples.append({"input": example["text"], "output": example_output})

INSTRUCTIONS = {
    "baseline": BASE_INSTRUCTION,
    "few_shot": BASE_INSTRUCTION + "\n\nEjemplos de resolución:\n" + json.dumps(examples, ensure_ascii=False),
}
SETTINGS = {"temperature": 0.0, "max_tokens": 1200}
TIMEOUT_SECONDS = 60
print(INSTRUCTIONS["few_shot"])

## 5. Ejecutar con PydanticAI

La llamada está a la vista: `Agent` recibe modelo, esquema e instrucción; `await agent.run(...)` envía solo el documento. Cada documento se procesa sin historial previo. Como máximo se permiten dos solicitudes por ejecución para que un error de esquema pueda reintentarse una vez; el cliente del proveedor puede realizar reintentos de transporte adicionales. El límite de tiempo también se conserva.

En modo simulado, `simulated_output` selecciona una salida preparada para comprobar el circuito. En modo real, `create_model` ignora ese argumento y consulta el proveedor configurado. Las salidas preparadas no garantizan que añadir ejemplos mejore un modelo real.

La imagen se adjunta como bytes, sin sustituirla por el texto ni por la referencia. El perfil real debe admitir imágenes. Si una consulta falla, queda registrada y cuenta como una extracción fallida en los denominadores; el resto del conjunto continúa. El modo simulado no aporta mediciones de tokens, costos ni latencia del proveedor.

In [ ]:
rows = []
for variant, instruction in INSTRUCTIONS.items():
    for case in cases:
        row = {"variant": variant, "case_id": case["id"], "document": case["document"],
               "model": model_ref, "mode": config.mode,
               "utc_time": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "status": "error", "output": None}
        started = perf_counter()
        try:
            path = (DATASET_PATH.parent / case["document"] if DATASET_PATH
                    else lab_resource(2, case["document"]))
            model = create_model(model_ref, config,
                                 simulated_output=case.get("prepared_outputs", {}).get(variant))
            agent = Agent(model, output_type=Extraction, instructions=instruction,
                          model_settings=SETTINGS, retries=1)
            result = await asyncio.wait_for(
                agent.run(["Extrae los campos del documento adjunto.", document_input(path)],
                          usage_limits=UsageLimits(request_limit=2)), TIMEOUT_SECONDS)
            row.update(status="ok", output=result.output.model_dump(mode="json"))
            if config.mode == "real":
                usage = result.usage
                row.update(input_tokens=usage.input_tokens, output_tokens=usage.output_tokens,
                           requests=usage.requests, **cost(model_ref, usage, config))
        except Exception as error:
            # El tipo basta para registrar el fallo sin imprimir cuerpos HTTP ni claves.
            row["error_type"] = type(error).__name__
        row["latency_ms"] = round((perf_counter() - started) * 1000) if config.mode == "real" else None
        rows.append(row)
        print(variant, case["id"], row["status"], row.get("error_type", ""))

## 6. Guardar ambas versiones

El archivo guarda instrucciones completas, esquema, configuración no secreta, referencias y salidas de ambas versiones. Una nueva ejecución sustituye este experimento completo: copia o versiona la evidencia anterior antes de volver a consultar si quieres conservar más iteraciones. Ninguna clave se guarda y los errores se registran sin el mensaje crudo del proveedor.

El siguiente paso relee este JSON y recalcula las métricas sin llamar al modelo. Así otra persona puede revisar el número sin tu clave. Conserva también los documentos originales de la muestra definitiva en el repositorio del equipo.

In [ ]:
evidence_path = ROOT / "labs/lab02/evidence/experiment.json"
payload = {
    "dataset": dataset["name"], "dataset_status": dataset["status"],
    "models": {MODEL_ROLE: model_ref}, "settings": SETTINGS,
    "output_schema": Extraction.model_json_schema(), "instructions": INSTRUCTIONS,
    "cases": [{key: case[key] for key in ("id", "document", "reference", "annotation_note")}
              for case in cases],
    "reference_reviewed": REFERENCE_REVIEWED, "results": rows,
    "decisions": {"change": "", "evidence": "", "remaining_error": ""},
}
save_experiment(evidence_path, payload, config)
print("Evidencia:", evidence_path)
print("Modo guardado:", config.mode, "· Referencia revisada:", REFERENCE_REVIEWED)

## 7. Recalcular exactitud y completitud por campo

Exactitud: valores iguales a la referencia divididos por todos los casos de ese campo, incluidos los fallidos. `null` frente a `null` es un acierto, si la salida cumple el esquema. La completitud solo cuenta los valores distintos de nulo: llenar más campos puede agregar invenciones. También se muestra cuántas salidas fallan o no cumplen el esquema.

No se mezclan tipos: un entero, una cadena y un booleano son valores diferentes para esta comprobación. Esta métrica no comprueba por sí sola que la cita sea fiel: inspecciona el respaldo en el original. La revisión humana debe incluir aciertos y errores, no solo los casos que el modelo considera dudosos.

In [ ]:
saved = load_json(evidence_path)
reference = {case["id"]: case["reference"] for case in saved["cases"]}
details = []
for row in saved["results"]:
    valid = row["status"] == "ok"
    output = {}
    if valid:
        try:
            output = Extraction.model_validate(row["output"]).model_dump(mode="json")
        except ValidationError:
            valid = False
    for field in FIELDS:
        extracted = output.get(field, {})
        predicted = extracted.get("value")
        expected = reference[row["case_id"]][field]
        correct = valid and type(predicted) is type(expected) and predicted == expected
        details.append({"variant": row["variant"], "case_id": row["case_id"], "field": field,
                        "expected": expected, "predicted": predicted, "correct": correct,
                        "completed": valid and predicted is not None, "failed": not valid,
                        "declared_confidence": extracted.get("declared_confidence"),
                        "evidence": extracted.get("evidence")})

detail = pd.DataFrame(details)
metrics = detail.groupby(["variant", "field"]).agg(
    correct=("correct", "sum"), cases=("correct", "size"),
    completed=("completed", "sum"), failed=("failed", "sum"),
)
metrics["accuracy"] = metrics["correct"] / metrics["cases"]
metrics["completeness"] = metrics["completed"] / metrics["cases"]
display(metrics)
display(detail.loc[~detail["correct"], ["variant", "case_id", "field", "expected", "predicted", "failed"]])

## 8. Contrastar la confianza y explicar el cambio

La siguiente tabla conserva los casos en que el modelo declara al menos 0,8 de confianza y muestra cuántos acierta. Es una inspección descriptiva de esta muestra, no una demostración de calibración ni una regla para automatizar acciones. En modo simulado todas las cifras de confianza fueron preparadas a propósito, incluidas las asociadas a errores.

Busca al menos un error en el original, identifica el respaldo que falta o contradice la salida y modifica solo una parte de la instrucción. Antes de otra consulta, versiona la evidencia anterior. Repite la comparación completa: una corrección puede introducir errores nuevos. Con el par texto/imagen, inspecciona si el cambio de formato altera el resultado.

In [ ]:
HIGH_CONFIDENCE = 0.8
confident = detail[detail["declared_confidence"] >= HIGH_CONFIDENCE]
if confident.empty:
    print("No hay campos con confianza declarada sobre el umbral elegido.")
else:
    confidence_check = confident.groupby("variant").agg(
        fields=("correct", "size"), correct=("correct", "sum"))
    confidence_check["observed_accuracy"] = confidence_check["correct"] / confidence_check["fields"]
    display(confidence_check)

# Completa con una decisión y una evidencia concreta del experimento de tu equipo.
DECISIONS = {"change": "", "evidence": "", "remaining_error": ""}
saved["decisions"] = DECISIONS
saved["reference_reviewed"] = REFERENCE_REVIEWED
save_experiment(evidence_path, saved, config)

## 9. Producto compartido y evidencia individual

En el repositorio del equipo: notebook, documentos de la muestra, `labs/lab02/evidence/experiment.json`, comparación por campo y explicación de una mejora o de una hipótesis que no mejoró. Conserva modelo, configuración y referencias. La etiqueta o commit permite congelar lo que se revisa. Solo lo validado se promueve a `src/`.

En el portafolio de cada integrante: reflexión, historial completo del asistente utilizado en la actividad y enlace al producto del equipo. Declara también si no usaste asistente. Retira datos sensibles y credenciales e indica las omisiones. Responde la pregunta individual de cinco minutos donde indique el docente; no se fija aquí un plazo ni un canal sin su acuerdo.

**Antes de presentar esto como evidencia real:** cambia a modo real, revisa la referencia, ejecuta con la muestra definitiva y completa las decisiones. Una ejecución simulada solo comprueba el circuito. Si falta configuración o falla el proveedor, registra el bloqueo; no reemplaces una medición real por una simulada sin indicarlo.

Fuentes técnicas del docente: [PydanticAI: salida estructurada](https://pydantic.dev/docs/ai/core-concepts/output/) y [entrada multimodal](https://pydantic.dev/docs/ai/core-concepts/input/), inglés, consulta de secciones de unos 10 minutos cada una, revisadas el 20/10/2026. El código se prueba por separado de la comprobación con proveedor real.